# Kolos: dish recognition — training on Google Colab

Runs steps 2–3 of `ml/foodphoto` (see `ml/README.md`) on a free Colab GPU.

**Before the first run**
1. *Runtime → Change runtime type → T4 GPU.*
2. Put `ua_photos.zip` (made locally by `python -m ml.foodphoto.pack`) into
   `My Drive/Kolos/foodphoto/` on Google Drive.

**Every session:** *Runtime → Run all.* Checkpoints are written to Google Drive after each epoch,
so if Colab disconnects, run everything again: training continues from the last finished epoch.

**At the end:** download `My Drive/Kolos/foodphoto/runs/` (without the `*.pt` files, except
`best.pt` of the model that goes into the app) and put it into `ml/models/foodphoto/` locally.

In [ ]:
# 1. GPU check: a T4 (or better) should be listed.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Google Drive (persistent storage) and paths.
import os
from google.colab import drive

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Kolos/foodphoto'
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
os.environ['FOODPHOTO_DATA'] = '/content/data'          # images: fast local disk
os.environ['FOODPHOTO_RUNS'] = f'{DRIVE}/runs'          # checkpoints and results: Drive
print(sorted(os.listdir(DRIVE)))

In [ ]:
# 3. Code from GitHub (branch phase-f2).
%cd /content
!test -d kolos && git -C kolos pull -q || git clone -q --depth 1 --branch phase-f2 https://github.com/dkrainii/course_project.git kolos
%cd /content/kolos
!git log --oneline -1

In [ ]:
# 4. Data. First session: download Food-101 (~5 GB), resize everything, cache the result on
#    Drive (~2 GB). Later sessions only unpack the cache (~2 min).
#    The last line must list 'ua': 879; otherwise read the message above it.
#    Runs in the notebook process, so an error stops "Run all" here.
from pathlib import Path
from ml.foodphoto import colab_data
colab_data.main(Path(DRIVE), workers=4)

## Training

Each run takes roughly 40–70 minutes on a T4. Run them in this order; if the session ends,
*Run all* again — finished runs are skipped in seconds and the interrupted one resumes.

In [ ]:
# 5. Main model: EfficientNet-B0, whole network fine-tuned.
!python -m ml.foodphoto.train --run effb0_finetune --workers 2

In [ ]:
# 6. Baseline: EfficientNet-B0 with a frozen backbone (only the classifier is trained).
!python -m ml.foodphoto.train --run effb0_probe --workers 2

In [ ]:
# 7. Lighter alternative: MobileNetV3-Large, whole network fine-tuned.
!python -m ml.foodphoto.train --run mnv3_finetune --workers 2

In [ ]:
# 8. Ablation: the main model without class-balanced sampling.
!python -m ml.foodphoto.train --run effb0_nobalance --workers 2

In [ ]:
# 9. Summary of what is on Drive.
import json, glob
for path in sorted(glob.glob(f'{DRIVE}/runs/*/metrics.json')):
    print(json.load(open(path)))